# 🧠 Session 2: Large Language Models & APIs
### Agentic AI Nano Bootcamp | Day 1 · Session 2

> **VS Code Setup:** Open this file with the [Jupyter extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter). Select your Python kernel via the kernel picker (top-right). A virtual environment with dependencies installed is recommended.

---

## 🎯 Learning Objectives
By the end of this session you will be able to:
- Understand how LLMs are architected (Transformers, attention)
- Set up and authenticate the OpenAI API
- Make API calls using different prompting techniques
- Access open-source models via Hugging Face
- Build a summarization agent and translation agent

## 📋 Session Outline
1. LLM Architecture: Transformers & Attention
2. Popular LLMs Compared
3. Setting Up the OpenAI API
4. Making API Calls — Prompting Techniques
5. Open-Source LLMs on Hugging Face
6. 🔬 Lab 1: Build a Summarization Agent
7. 🔬 Lab 2: Build a Translation Agent


## 🛠️ VS Code Quick-Start

1. **Install the Jupyter extension** — search `ms-toolsai.jupyter` in the Extensions panel.
2. **Select a kernel** — click the kernel picker (top-right of the notebook) → *Python Environments* → choose your venv or base interpreter.
3. **Create a `.env` file** in the same folder as this notebook:
   ```
   OPENAI_API_KEY=sk-...
   ```
4. **Run cells** with `Shift+Enter` (run & advance) or `Ctrl+Enter` (run in place).
5. **Restart kernel** via the ↺ button in the toolbar if you install new packages.

> 💡 **Tip:** The first cell (`Environment Setup`) installs all Python packages automatically. Run it once, then restart the kernel.


## 1. LLM Architecture: Transformers & Attention

### The Transformer (Vaswani et al., 2017)

The transformer architecture is the foundation of every modern LLM. It replaced RNNs and LSTMs by processing **all tokens in parallel** rather than sequentially.

### Key Components

```
Input Text → Tokenizer → Token Embeddings → Positional Encoding
                                                    ↓
                                    ┌───────────────────────────┐
                                    │  Transformer Block × N    │
                                    │  ┌─────────────────────┐  │
                                    │  │ Multi-Head Attention │  │
                                    │  └─────────────────────┘  │
                                    │  ┌─────────────────────┐  │
                                    │  │  Feed-Forward Layer  │  │
                                    │  └─────────────────────┘  │
                                    │  ┌─────────────────────┐  │
                                    │  │    Layer Norm        │  │
                                    │  └─────────────────────┘  │
                                    └───────────────────────────┘
                                                    ↓
                                        Output Token Probabilities
```

### Attention: The Core Innovation

Attention allows every token to "look at" every other token and decide how much to focus on it:

```
"The animal didn't cross the street because it was too tired"
                                          ↑
     When predicting what 'it' refers to, attention mechanism
     gives high weight to 'animal' and low weight to 'street'
```

### Key Parameters That Define an LLM

| Parameter | What it means | Example |
|---|---|---|
| **Parameters** | Learnable weights | 7B, 70B, 175B |
| **Context window** | Max tokens at once | 8K, 128K, 1M |
| **Temperature** | Randomness of output | 0.0 (deterministic) – 2.0 (creative) |
| **Top-p** | Nucleus sampling threshold | 0.9 = consider top 90% probability mass |
| **Max tokens** | Limit on output length | 1 – 4096 |

In [ ]:
# ── Tokenization Demo (no API key needed) ──
# pip install tiktoken

try:
    import tiktoken
    HAS_TIKTOKEN = True
except ImportError:
    HAS_TIKTOKEN = False
    print('Installing tiktoken...')
    import subprocess
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'tiktoken', '-q'])
    import tiktoken
    HAS_TIKTOKEN = True

enc = tiktoken.encoding_for_model("gpt-4")

examples = [
    "Hello, world!",
    "The transformer architecture revolutionized NLP.",
    "ChatGPT is built on GPT-4o which uses RLHF fine-tuning.",
    "நான் தமிழ் பேசுகிறேன்",   # Tamil text
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)"
]

print('🔤 Tokenization Examples')
print('=' * 70)
for text in examples:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f'\nText:   {text[:60]}')
    print(f'Tokens: {decoded}')
    print(f'Count:  {len(tokens)} tokens')

print('\n💡 Notice: non-English text uses more tokens per character!')
print('💡 Code tokens map closely to programming keywords.')

## 2. Popular LLMs Compared

| Model | Creator | Params | Context | Open? | Strength |
|---|---|---|---|---|---|
| **GPT-4o** | OpenAI | ~1.8T (est.) | 128K | ❌ | All-round, multimodal |
| **Claude 3.5 Sonnet** | Anthropic | Unknown | 200K | ❌ | Long context, safety |
| **Gemini 1.5 Pro** | Google | Unknown | 1M | ❌ | Ultra-long context |
| **Llama 3 70B** | Meta | 70B | 8K | ✅ | Best open-source |
| **Mistral 7B** | Mistral AI | 7B | 32K | ✅ | Efficient, fast |
| **BERT** | Google | 110M–340M | 512 | ✅ | Embeddings, classification |

### When to Use Which Model?

```
Complex reasoning / creative tasks  →  GPT-4o, Claude 3.5
Very long documents                 →  Claude 3.5 (200K), Gemini 1.5 (1M)
Production / cost-sensitive         →  GPT-4o-mini, Mistral 7B
Self-hosted / private data          →  Llama 3, Mistral
Text embeddings / semantic search   →  text-embedding-3-small, BERT
```

## 3. Setting Up the OpenAI API

### Step-by-step Setup

**Step 1:** Get your API key
- Go to https://platform.openai.com/api-keys
- Click **"Create new secret key"**
- Copy the key — you only see it once!

**Step 2:** Install the library
```bash
pip install openai python-dotenv
```

**Step 3:** Create a `.env` file in the same folder as this notebook
```
OPENAI_API_KEY=sk-...
```
Add `.env` to your `.gitignore` — never commit secrets!

**Step 4 (alternative):** Set the key in your shell before launching VS Code
```bash
# Mac/Linux
export OPENAI_API_KEY='sk-...'

# Windows PowerShell
$env:OPENAI_API_KEY = 'sk-...'
```

> ⚠️ **Security rule**: Never hardcode your API key. Use a `.env` file or environment variable.


In [ ]:
# ── Environment Setup ──
# Run this cell first to install dependencies and load your API key.

import subprocess, sys, os

# Install dependencies into the current kernel
print('📦 Installing dependencies...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'openai', 'python-dotenv', 'tiktoken', '-q'],
    check=True
)
print('✅ Dependencies installed')

# Load .env file if present (place it next to this notebook)
from dotenv import load_dotenv
load_dotenv(override=False)  # won't overwrite an already-set env var

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f'✅ API key loaded: sk-...{api_key[-4:]}')
else:
    print('⚠️  OPENAI_API_KEY not found.')
    print('   Create a .env file in this folder with: OPENAI_API_KEY=sk-...')

from openai import OpenAI
client = OpenAI()  # automatically reads OPENAI_API_KEY from environment
print('\n✅ OpenAI client initialized')


In [ ]:
# ── Your First API Call ──

response = client.chat.completions.create(
    model="gpt-4o-mini",       # Cost-efficient model for learning
    messages=[
        {"role": "user", "content": "Explain what a Large Language Model is in 2 sentences."}
    ],
    max_tokens=100,
    temperature=0.7
)

# Extract the response text
answer = response.choices[0].message.content
print('🤖 Model Response:')
print('-' * 50)
print(answer)

# Inspect the full response object
print('\n📊 API Response Metadata:')
print(f'  Model:             {response.model}')
print(f'  Prompt tokens:     {response.usage.prompt_tokens}')
print(f'  Completion tokens: {response.usage.completion_tokens}')
print(f'  Total tokens:      {response.usage.total_tokens}')
print(f'  Finish reason:     {response.choices[0].finish_reason}')

## 4. Making API Calls — Prompting Techniques

### Understanding the Messages Array

OpenAI's chat API uses a `messages` list with three role types:

| Role | Purpose | Example |
|---|---|---|
| `system` | Set the assistant's persona and rules | "You are a senior telecom engineer..." |
| `user` | The human's input | "Why is my internet slow?" |
| `assistant` | Previous AI responses (for multi-turn) | "Let me check..." |

In [ ]:
# ── System Prompts: Giving the Model a Persona ──

def call_llm(system_prompt, user_message, model="gpt-4o-mini", temperature=0.7, max_tokens=300):
    """Reusable wrapper for OpenAI chat completions."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

# Same question, 3 different personas
question = "What is 5G?"

personas = [
    ("You are a friendly kindergarten teacher. Use simple words and an analogy.",
     "👶 Kindergarten Teacher"),
    ("You are a senior telecom engineer. Use precise technical terminology.",
     "👷 Telecom Engineer"),
    ("You are a stand-up comedian. Make it funny but accurate.",
     "😄 Comedian"),
]

for system, label in personas:
    answer = call_llm(system, question)
    print(f'\n{label}:')
    print('-' * 50)
    print(answer)

In [ ]:
# ── Effect of Temperature on Output ──
# Temperature controls randomness: 0 = deterministic, 2 = very random

prompt = "Write a one-sentence tagline for a new AI-powered telecom product."

print('🌡️  Effect of Temperature on LLM Output')
print('=' * 60)

for temp in [0.0, 0.5, 1.0, 1.5]:
    result = call_llm(
        "You write creative product taglines.",
        prompt,
        temperature=temp,
        max_tokens=60
    )
    label = {0.0: 'Deterministic', 0.5: 'Balanced', 1.0: 'Creative', 1.5: 'Very creative'}[temp]
    print(f'\nTemp {temp} ({label}):')
    print(f'  {result}')

print('\n💡 Run this cell multiple times — temp=0.0 always gives the same result!')

In [ ]:
# ── Multi-Turn Conversation ──
# The model has no memory — we send the full conversation history each time

def chat(conversation_history, user_message, system="You are a helpful AI assistant."):
    """Appends user message and gets response, maintaining conversation history."""
    conversation_history.append({"role": "user", "content": user_message})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system}] + conversation_history,
        temperature=0.7,
        max_tokens=200
    )
    
    assistant_message = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_message})
    return assistant_message, conversation_history

# Simulate a multi-turn support conversation
history = []
system_prompt = "You are a telecom customer support agent. Be concise and helpful."

turns = [
    "Hi, my internet has been slow since yesterday morning.",
    "I'm on a 100 Mbps plan but getting only 5 Mbps.",
    "Yes I tried restarting. Lights on router all green.",
]

print('💬 Multi-Turn Support Conversation')
print('=' * 60)

for turn in turns:
    print(f'\n👤 Customer: {turn}')
    response, history = chat(history, turn, system=system_prompt)
    print(f'🤖 Agent:    {response}')

print(f'\n📊 Total messages in history: {len(history)}')

## 5. Open-Source LLMs on Hugging Face

### Why Open-Source LLMs?

| Consideration | Proprietary (OpenAI) | Open-Source (HF) |
|---|---|---|
| **Cost** | Per-token billing | Free to run locally |
| **Privacy** | Data sent to OpenAI | Runs on your hardware |
| **Customization** | Limited fine-tuning | Full control |
| **Latency** | Network dependent | Local = fast |
| **Setup** | API key only | Hardware requirements |

### Popular Hugging Face Models

```python
# BERT — for embeddings and classification
"bert-base-uncased"           # 110M params, English
"bert-base-multilingual-cased" # 179M params, 104 languages

# Mistral — for text generation
"mistralai/Mistral-7B-Instruct-v0.2"   # 7B params, instruction-tuned

# Summarization
"facebook/bart-large-cnn"    # Great for news summarization
"google/pegasus-xsum"        # Abstractive summarization

# Translation
"Helsinki-NLP/opus-mt-en-fr" # English → French
"Helsinki-NLP/opus-mt-en-ROMANCE" # English → all Romance languages
```

In [ ]:
# ── Hugging Face Pipeline Demo ──
# Uses the 'transformers' pipeline API — easiest way to use HF models

import subprocess, sys
print('📦 Installing transformers (this may take a minute)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', 'torch', 'sentencepiece', '-q'], capture_output=True)
print('✅ Ready')

from transformers import pipeline

# ── Sentiment Analysis using BERT ──
print('\n🎭 Sentiment Analysis (BERT-based):')
print('-' * 50)
sentiment = pipeline("sentiment-analysis")  # downloads distilbert by default

texts = [
    "The new 5G plan is amazing, speeds are incredible!",
    "My connection drops every hour. Worst service ever.",
    "The customer support resolved my issue eventually."
]

for text in texts:
    result = sentiment(text)[0]
    emoji = '😊' if result['label'] == 'POSITIVE' else '😞'
    print(f'{emoji} [{result["label"]:8}  {result["score"]:.2%}]  {text[:60]}')

In [ ]:
# ── Summarization using BART (Hugging Face, no API key needed) ──

from transformers import pipeline

print('📥 Loading BART summarization model (first run downloads ~1.6GB)...')
summarizer_hf = pipeline("summarization", model="facebook/bart-large-cnn")
print('✅ Model loaded')

article = """
5G technology represents a major leap forward in wireless communication, 
offering speeds up to 100 times faster than 4G LTE. With latency as low as 
1 millisecond, 5G enables real-time applications that were previously impossible 
on mobile networks. The technology operates across three frequency bands: 
low-band for coverage, mid-band for the balance of speed and range, and 
millimeter wave for ultra-high speeds in dense urban areas. Beyond smartphones, 
5G is driving the Internet of Things revolution, connecting billions of devices 
in smart cities, autonomous vehicles, remote surgery, and industrial automation. 
Major telecom operators worldwide are investing hundreds of billions of dollars 
to build out 5G infrastructure, with global coverage expected to reach 60% of 
the world population by 2026.
"""

summary = summarizer_hf(article.strip(), max_length=80, min_length=30, do_sample=False)

print('\n📄 Original Article:')
print(article.strip())
print(f'\n📝 BART Summary ({len(summary[0]["summary_text"].split())} words):')
print(summary[0]['summary_text'])
print(f'\n📊 Compression ratio: {len(article.split()):.0f} → {len(summary[0]["summary_text"].split())} words')

## 🔬 Lab 1: Build a Summarization Agent

We will build a **Summarization Agent** that:
- Accepts any text as input
- Generates summaries at different levels (brief / detailed / bullets)
- Extracts key entities and action items
- Uses the OpenAI API under the hood

In [ ]:
# ── Lab 1: Summarization Agent ──

from openai import OpenAI
import os

client = OpenAI()

class SummarizationAgent:
    """
    An agent that summarizes text in multiple styles.
    """
    
    def __init__(self, model="gpt-4o-mini"):
        self.model  = model
        self.client = OpenAI()
        self.system = (
            "You are an expert summarization assistant. "
            "Your summaries are accurate, concise, and preserve key facts."
        )
    
    def _call(self, prompt, max_tokens=400):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system},
                {"role": "user",   "content": prompt}
            ],
            temperature=0.3,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    
    def brief(self, text):
        """One-sentence summary."""
        return self._call(f"Summarize the following in exactly one sentence:\n\n{text}")
    
    def detailed(self, text, sentences=3):
        """Multi-sentence detailed summary."""
        return self._call(
            f"Summarize the following in {sentences} sentences, "
            f"preserving all key facts:\n\n{text}"
        )
    
    def bullet_points(self, text, n=5):
        """Extract key points as a bulleted list."""
        return self._call(
            f"Extract the {n} most important points from the following "
            f"as a bulleted list (use • as bullet symbol):\n\n{text}"
        )
    
    def action_items(self, text):
        """Extract action items and decisions."""
        return self._call(
            f"Extract all action items, decisions, and next steps from the following. "
            f"If none exist, say 'No action items found.':\n\n{text}"
        )
    
    def key_entities(self, text):
        """Extract key people, organizations, dates, numbers."""
        return self._call(
            f"Extract key entities from the text (people, organizations, dates, "
            f"locations, numbers). Format as: Entity: Type\n\n{text}"
        )
    
    def summarize_all(self, text):
        """Run all summarization modes and display results."""
        print('📋 SUMMARIZATION AGENT REPORT')
        print('=' * 60)
        print(f'📄 Input ({len(text.split())} words):')
        print(text[:200] + ('...' if len(text) > 200 else ''))
        print()
        
        modes = [
            ('1️⃣  One-line summary',  self.brief),
            ('2️⃣  Detailed summary',  self.detailed),
            ('3️⃣  Bullet points',     self.bullet_points),
            ('4️⃣  Action items',      self.action_items),
            ('5️⃣  Key entities',      self.key_entities),
        ]
        
        for label, fn in modes:
            print(f'{label}:')
            print('-' * 40)
            print(fn(text))
            print()

print('✅ SummarizationAgent class defined')

In [ ]:
# ── Run the Summarization Agent ──

# Sample input: a realistic telecom incident report
incident_report = """
INCIDENT REPORT — Network Outage: Chennai North Zone
Date: April 14, 2025 | Reported By: NOC Team | Priority: P1

At 14:32 IST, a major outage was detected affecting approximately 45,000 subscribers 
in the Chennai North zone covering Perambur, Tondiarpet, and Kolathur areas. 
The outage was triggered by a hardware failure in the primary BTS (Base Transceiver Station) 
at the Perambur node, compounded by a failed automatic failover to the backup unit.

Initial investigation by field engineer Ramesh Kumar (ID: TN-2341) identified 
a power supply unit (PSU) failure combined with a firmware bug in version 4.2.1 
that prevented the redundancy switch. The issue escalated to Level-3 support at 15:00 IST. 
Service was partially restored by 16:45 IST following a manual failover. 
Full restoration was completed at 18:20 IST after replacement of the PSU unit.

Total downtime: 3 hours 48 minutes. Estimated customer impact: 45,000 subscribers. 
SLA breach confirmed — penalty clause activated per contract. 
Root cause: Hardware failure + firmware bug #4821 in BTS firmware v4.2.1.

Action required:
1. Roll back firmware v4.2.1 across all nodes in Chennai zone by April 16
2. Conduct emergency audit of all PSU units older than 3 years
3. Submit RCA (Root Cause Analysis) to regulator within 72 hours
4. Compensate affected subscribers with 3 days of free service
5. Schedule vendor call with BTS manufacturer for firmware patch by April 20
"""

agent = SummarizationAgent()
agent.summarize_all(incident_report)

In [ ]:
# ── Try it yourself! ──
# Paste any text below and run the agent

your_text = """
Paste your own text here — a news article, a meeting transcript, 
an email, or any document you want summarized.
"""

agent = SummarizationAgent()

# Try individual modes:
print('📌 One-line summary:')
print(agent.brief(your_text))

print('\n📌 Bullet points:')
print(agent.bullet_points(your_text, n=3))

## 🔬 Lab 2: Build a Translation Agent

We will build a **Translation Agent** that:
- Detects the source language automatically
- Translates to one or multiple target languages
- Preserves domain-specific terminology (telecom, medical, legal)
- Supports batch translation

In [ ]:
# ── Lab 2: Translation Agent ──

class TranslationAgent:
    """
    A multi-language translation agent with domain awareness.
    """
    
    SUPPORTED_LANGUAGES = {
        'en': 'English', 'ta': 'Tamil', 'hi': 'Hindi', 'fr': 'French',
        'de': 'German', 'es': 'Spanish', 'ar': 'Arabic', 'ja': 'Japanese',
        'zh': 'Chinese (Simplified)', 'pt': 'Portuguese'
    }
    
    def __init__(self, model="gpt-4o-mini"):
        self.model  = model
        self.client = OpenAI()
    
    def _call(self, prompt, max_tokens=500):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content.strip()
    
    def detect_language(self, text):
        """Detect the language of the input text."""
        result = self._call(
            f"What language is this text written in? "
            f"Respond with ONLY the language name and ISO 639-1 code like: English (en)\n\n{text}"
        )
        return result
    
    def translate(self, text, target_lang, domain=None):
        """Translate text to target language, optionally with domain context."""
        target_name = self.SUPPORTED_LANGUAGES.get(target_lang, target_lang)
        domain_note = f" This is a {domain} context — preserve technical terminology." if domain else ""
        
        prompt = (
            f"Translate the following text to {target_name}.{domain_note} "
            f"Return ONLY the translation, no explanations.\n\n{text}"
        )
        return self._call(prompt)
    
    def translate_multi(self, text, target_langs, domain=None):
        """Translate to multiple languages at once."""
        target_names = [self.SUPPORTED_LANGUAGES.get(l, l) for l in target_langs]
        langs_str = ', '.join(target_names)
        domain_note = f" This is a {domain} context — preserve technical terminology." if domain else ""
        
        prompt = (
            f"Translate the following text into each of these languages: {langs_str}.{domain_note}\n"
            f"Format your response as:\n"
            f"LANGUAGE: translation\n\n"
            f"Text to translate:\n{text}"
        )
        return self._call(prompt, max_tokens=800)
    
    def back_translate_check(self, original, target_lang):
        """Translate to target, then back to original to check quality."""
        translated  = self.translate(original, target_lang)
        back        = self.translate(translated, 'en')
        
        print(f'Original:           {original}')
        print(f'Translated ({target_lang}): {translated}')
        print(f'Back-translated:    {back}')
        
        # Ask model to assess quality
        quality = self._call(
            f"Rate the semantic similarity between these two English sentences on a scale of 1-10 "
            f"and explain in one sentence:\n"
            f"Original: {original}\nBack-translated: {back}"
        )
        print(f'Quality check:      {quality}')
        return translated

print('✅ TranslationAgent class defined')

In [ ]:
# ── Run the Translation Agent ──

agent = TranslationAgent()

# 1. Language Detection
print('🔍 Language Detection:')
print('-' * 50)
samples = [
    "The network outage affected 45,000 subscribers.",
    "இணைய இணைப்பு துண்டிக்கப்பட்டது.",          # Tamil
    "La connexion réseau est interrompue.",          # French
    "नेटवर्क कनेक्शन टूट गया है।",                 # Hindi
]
for sample in samples:
    print(f'  "{sample[:50]}" → {agent.detect_language(sample)}')

# 2. Domain-aware translation (telecom)
print('\n🌐 Domain-Aware Translation (Telecom):')
print('-' * 50)
telecom_text = "The BTS experienced a PSU failure causing a 4-hour outage in the LTE band."
print(f'Original: {telecom_text}')
print(f'Tamil:    {agent.translate(telecom_text, "ta", domain="telecom")}')
print(f'Hindi:    {agent.translate(telecom_text, "hi", domain="telecom")}')

# 3. Multi-language translation
print('\n🌍 Multi-Language Translation:')
print('-' * 50)
msg = "Your internet service has been restored. We apologize for the inconvenience."
print(f'Original: {msg}\n')
result = agent.translate_multi(msg, ['ta', 'hi', 'fr', 'de'])
print(result)

In [ ]:
# ── Batch Translation: Customer Notifications ──
# A practical telecom use case: translate outage SMS notifications

agent = TranslationAgent()

notifications = [
    "Dear customer, planned maintenance on April 16 from 2-4 AM may affect your service.",
    "Your data usage has reached 80% of your monthly limit. Recharge to avoid slowdowns.",
    "Your issue has been resolved. Please rate our service on a scale of 1 to 5."
]

target_languages = ['ta', 'hi']  # Tamil and Hindi for Indian market

print('📱 Batch SMS Translation for Telecom Notifications')
print('=' * 60)

for i, notif in enumerate(notifications, 1):
    print(f'\n[Notification {i}]')
    print(f'EN: {notif}')
    for lang in target_languages:
        translated = agent.translate(notif, lang, domain='telecom customer service')
        print(f'{lang.upper()}: {translated}')

print('\n✅ Batch translation complete!')

## ✅ Session 2 Summary

| Concept | Key Takeaway |
|---|---|
| **Transformer** | Foundation of all LLMs — parallel attention over tokens |
| **Tokenization** | Text is split into tokens; non-English uses more tokens |
| **API setup** | Use environment variables; never hardcode API keys |
| **Messages array** | system + user + assistant roles for full context control |
| **Temperature** | 0 = deterministic, 1+ = creative |
| **Open-source LLMs** | Hugging Face pipelines — free, private, customizable |
| **Agents** | Wrap API calls in classes with multiple capabilities |

## 🔗 What's Next
**Session 3** covers **Prompt Engineering** — the art of writing prompts that reliably get the output you want. We will learn zero-shot, few-shot, chain-of-thought, and more.

---
*Agentic AI Nano Bootcamp | Day 1 Session 2*